In [6]:
!pip install -U transformers
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 66.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def lower(text):
    return text.lower()

a = "Testing lowercase"
b = lower(a)
print(b)

testing lowercase


In [8]:
sentences = [
    "I love this!",
    "Yeh movie bahut bakwaas thi",
    "Bro this rap is fire",
    "Mast vibe aa rahi hai",
    "Worst experience ever."
]

for s in sentences:
    tokens = tokenizer.tokenize(s)
    ids = tokenizer(s)["input_ids"]
    print("\nSentence:", s)
    print("Lowercased Sentence:", s)
    print("Tokens:", tokens)
    print("IDs:", ids)



Sentence: I love this!
Lowercased Sentence: I love this!
Tokens: ['▁I', '▁love', '▁this', '!']
IDs: [0, 87, 5161, 903, 38, 2]

Sentence: Yeh movie bahut bakwaas thi
Lowercased Sentence: Yeh movie bahut bakwaas thi
Tokens: ['▁Yeh', '▁movie', '▁bahut', '▁bak', 'wa', 'as', '▁thi']
IDs: [0, 54611, 14277, 164433, 3472, 634, 162, 6117, 2]

Sentence: Bro this rap is fire
Lowercased Sentence: Bro this rap is fire
Tokens: ['▁Bro', '▁this', '▁rap', '▁is', '▁fire']
IDs: [0, 13177, 903, 29892, 83, 11476, 2]

Sentence: Mast vibe aa rahi hai
Lowercased Sentence: Mast vibe aa rahi hai
Tokens: ['▁Mas', 't', '▁vi', 'be', '▁a', 'a', '▁rahi', '▁hai']
IDs: [0, 3010, 18, 279, 372, 10, 11, 125672, 1337, 2]

Sentence: Worst experience ever.
Lowercased Sentence: Worst experience ever.
Tokens: ['▁Wor', 'st', '▁experience', '▁ever', '.']
IDs: [0, 145062, 271, 16981, 17669, 5, 2]


In [9]:
import re
def clean_special(text):
    text = re.sub(r"http\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"#(\w+)", r"\1", text)  # keep word, drop '#'
    return text


In [10]:
!pip install emoji
import emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.7 MB/s eta 0:00:00


In [11]:
def emoji_to_text(text):
    return emoji.demojize(text, delimiters=(" ", " "))
print(emoji_to_text("🙏🙏🙏"))

 folded_hands  folded_hands  folded_hands 


In [12]:
def normalize_stretch(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)
normalize_stretch("goooooood")


'good'

In [13]:
!pip install indic-transliteration
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 5.1 MB/s eta 0:00:00


In [14]:
def translit_hinglish(text):
    # naive rule: try transliterating only pure-ish Hindi words
    words = text.split()
    new_words = []
    for w in words:
        try:
            # try converting to Devanagari
            new_words.append(transliterate(w, sanscript.ITRANS, sanscript.DEVANAGARI))
        except:
            new_words.append(w)
    return " ".join(new_words)

print(translit_hinglish("Inki suvidha acchi hai"))

ईन्कि सुविध अच्चि है


In [15]:
def preprocess(text):
    text = lower(text)
    text = clean_special(text)
    text = emoji_to_text(text)
    text = normalize_stretch(text)
    # toggle this depending on experiments:
    #text = translit_hinglish(text)
    return text.strip()

In [16]:
sample = "Brooooo yeh movie masttt hai 😂🔥 link dekho https://yt.com/xyz"
print(preprocess(sample))

broo yeh movie mastt hai  face_with_tears_of_joy  fire  link dekho  URL


In [17]:
import random

data = [
    ("I love this phone", 1),
    ("Worst experience ever", 0),
    ("This is okay, nothing special", 0),
    ("Yeh movie bahut mast thi", 1),
    ("Bohot bakwaas acting", 0),
    ("The service was fine", 1),
    ("Kya zabardast performance hai", 1),
    ("Paise waste ho gaye", 0),
    ("Not bad, not great", 1),
    ("Full paisa vasool", 1),
    ("crazy movie 🤌🏻🫶🏻🫶🏻🫶🏻", 1),
    ("Don't you dare to listen to the negativity out there.. just go and watch this raw and intense stuff 🔥😂", 1),
    ("Paisa Vasool Movie Hai Yeh, Akshay Khanna Best Acting, 👌👌👌", 1),
    ("The movie is too slow", 0),
    ("A Misguided Spy Drama That Confuses Noise with Narrative", 1),
    ("Complete waste of time dragged story", 0),
    ("An blood-crawling, gut wrenching, exhilarating movie!", 1),
    ("Abusive Language", 0),
    ("Undisputed King of Indian Action Spy Thrillers", 1),
    ("A Brutal And Unflinching War Drama That Loses Its Grip Along the Way", 1),
    ("New school Action unleashed with a old script", 1),
    ("Bigger Bolder but not better movie from Dhar", 1),
    ("Agar app hai Lints vagerah se pareshaan to iss Lint remover ko nahi kar paayenge inkaar", 1),
    ("Not producing Heat", 0),
    ("Okayish", 1),
    ("No space to cook very small", 0),
    ("Worth every penny❤️👍", 1),
    ("Okay okay", 1),
    ("It’s not even one month of purchase and charge stop working.", 0),
    ("unable to listen voice", 0),
    ("Good but ok", 1),
    ("Voice is very low, ring tone also low", 0),
    ("no help from customer service", 0),
    ("Good one but a bit costly..", 1),
    ("FANTASTIC and PHENOMENAL,BEAST PERFORMANCE 💪🏼", 1),
    ("What a fantastic all-rounder and a complete package.", 1),
    ("Value for Money Flagship Experience", 1),
    ("Powerful phone that is not the best!", 1),
    ("People like me who are always concerned about the battery charging will have a breather. Camera is a little bummer though", 1),
    ("Loooonnnng battery backup, fastest processor, smoothest OS i have ever used.", 1),
    ("Full paisa vasool bhai, maza aa gaya", 1),
    ("Ekdum solid performance, loved it", 1),
    ("Camera quality is amazing, totally worth it", 1),
    ("Bhai kya zabardast movie thi 🔥", 1),
    ("Smooth experience from start to end", 1),
    ("Sound quality is top notch, goosebumps", 1),
    ("Worth every rupee I spent on this", 1),
    ("Acting was powerful and impactful", 1),
    ("Crazy good performance, didn’t expect this", 1),
    ("Loved the vibe and the background music", 1),
    ("Mast direction, ekdum engaging", 1),
    ("This exceeded all my expectations", 1),
    ("Visuals are stunning and very immersive", 1),
    ("Ekdum badiya product, no complaints", 1),
    ("Fast, smooth, and reliable performance", 1),
    ("Dialogue delivery was on point", 1),
    ("Music hits hard, repeat worthy", 1),
    ("Paise waste ho gaye, bilkul bekaar", 0),
    ("Started well but ended very badly", 0),
    ("Camera is good but battery is horrible", 0),
    ("Bhai kya bakwaas experience tha", 0),
    ("Not worth the money at all", 0),
    ("Slow performance ruined everything", 0),
    ("Acting was forced and annoying", 0),
    ("Too much hype for such a weak product", 0),
    ("Movie drags a lot in the second half", 0),
    ("Expected a lot more, totally disappointed", 0),
    ("Worst experience I’ve had recently", 0),
    ("Lag hi nahi raha effort dala hai", 0),
    ("Screen is fine but performance sucks", 0),
    ("Direction was confusing and boring", 0),
    ("Overpriced and underdelivers badly", 0),
    ("Dialogues felt cringey and unnatural", 0),
    ("Nothing works properly, very frustrating", 0),
    ("Poor execution spoiled the idea", 0),
    ("Complete waste of time and money", 0),
    ("Sound quality is bad and irritating", 0),
    ("Story had potential but execution failed", 0),
    ("Bhai patience test ho gaya dekh ke", 0),
    ("Very dull and predictable", 0),
    ("Bad experience overall, not recommended", 0),
    ("Looks good but works terribly", 0),
    ("Performance issues throughout", 0),
    ("Totally misleading marketing", 0),
    ("Disappointing and forgettable", 0),
    ("Didn’t enjoy it even a little", 0),
    ("Regret spending money on this", 0)
]

random.shuffle(data)


In [18]:
import pandas as pd

df = pd.DataFrame(data, columns=["text", "label"])
df


,text,label
0,A Brutal And Unflinching War Drama That Loses ...,1
1,A Misguided Spy Drama That Confuses Noise with...,1
2,"Camera quality is amazing, totally worth it",1
3,"Voice is very low, ring tone also low",0
4,Overpriced and underdelivers badly,0
...,...,...
82,Okayish,1
83,Worst experience ever,0
84,"Ekdum solid performance, loved it",1
85,Too much hype for such a weak product,0


In [19]:
df["clean_text"] = df["text"].apply(preprocess)
df

,text,label,clean_text
0,A Brutal And Unflinching War Drama That Loses ...,1,a brutal and unflinching war drama that loses ...
1,A Misguided Spy Drama That Confuses Noise with...,1,a misguided spy drama that confuses noise with...
2,"Camera quality is amazing, totally worth it",1,"camera quality is amazing, totally worth it"
3,"Voice is very low, ring tone also low",0,"voice is very low, ring tone also low"
4,Overpriced and underdelivers badly,0,overpriced and underdelivers badly
...,...,...,...
82,Okayish,1,okayish
83,Worst experience ever,0,worst experience ever
84,"Ekdum solid performance, loved it",1,"ekdum solid performance, loved it"
85,Too much hype for such a weak product,0,too much hype for such a weak product


In [20]:
def weak_label_binary(text):
    text_lower = text.lower()

    negative_keywords = [
        "😡", "pouting_face", "bakwaas", "worst", "waste", "slow",
        "dragged", "abusive", "not working", "stop working", "unable",
        "low", "no help", "no space", "small", "not producing",
        "horrible", "bad experience", "negativity", "bummer", "bad",
        "costly", "loses its grip", "complete waste", "misguided",
        "pareshaan", "worst", "terrible", "horrible", "awful", "pathetic",
    "disappointing", "useless", "waste", "wasted", "overpriced",
    "slow", "laggy", "buggy", "broken", "crash",
    "poor", "bad", "boring", "annoying", "irritating",
    "bakwaas", "bekaar", "ghatiya", "faltu", "paise waste",
    "time waste", "bilkul bekaar", "maza nahi aaya",
    "pak gaya", "sir dard", "pareshaan", "not worth", "no value", "doesn't work", "not working",
    "stopped working", "failed", "problem", "issue", "😡", "🤬", "🤮", "😤", "💩"

    ]
    for keyword in negative_keywords:
        if keyword in text_lower:
            return 0

    positive_keywords = [
        "🔥", "fire", "mast", "love", "paisa vasool", "crazy",
        "revolving_hearts", "ok_hand", "best", "zabardast", "exhilarating",
        "undisputed king", "worth", "red_heart", "thumbs_up", "fantastic",
        "phenomenal", "beast performance", "flexed_biceps", "complete package",
        "value for money", "long battery", "fastest processor", "smoothest os",
        "superb", "amazing", "excellent", "great", "intense",
        "performance", "full paisa", "powerful", "vasool", "raw" "amazing", "excellent", "fantastic", "awesome", "brilliant",
    "outstanding", "perfect", "love", "loved", "enjoyed",
    "smooth", "fast", "powerful", "reliable", "impressive",
    "mast", "badiya", "zabardast", "paisa vasool", "full paisa vasool",
    "maza aa gaya", "dil jeet liya", "ekdum solid", "bawal",
    "next level", "level hi alag", "jhakaas",
    "worth it", "value for money", "highly recommend",
    "must watch", "must buy", "satisfied", "happy with",

    "🔥", "❤️", "😍", "👏", "💯"
    ]
    for keyword in positive_keywords:
        if keyword in text_lower:
            return 1

    return None

In [21]:
df["weak_label"] = df["text"].apply(weak_label_binary)
df = df[df["weak_label"].notna()]


In [22]:
auto_data = [
    "bhai kya mast gaana hai 🔥🔥",
    "yeh update ekdum bekaar hai 😡",
    "thik hi laga mujhe",
    "customer support was horrible",
    "best product ever bought ❤️",
    "it's okay, not bad not great",
    "amazing experience with this new feature",
    "complete waste of my time and money",
    "the battery life is superb",
    "installation was a little bummer",
    "Battery is very bad"
]

weak_df = pd.DataFrame({
    "text": auto_data,
})
weak_df["label"] = weak_df["text"].apply(weak_label_binary)
weak_df["clean_text"] = weak_df["text"].apply(preprocess)
weak_df

,text,label,clean_text
0,bhai kya mast gaana hai 🔥🔥,1.0,bhai kya mast gaana hai fire fire
1,yeh update ekdum bekaar hai 😡,0.0,yeh update ekdum bekaar hai enraged_face
2,thik hi laga mujhe,NaN,thik hi laga mujhe
3,customer support was horrible,0.0,customer support was horrible
4,best product ever bought ❤️,1.0,best product ever bought red_heart
5,"it's okay, not bad not great",0.0,"it's okay, not bad not great"
6,amazing experience with this new feature,1.0,amazing experience with this new feature
7,complete waste of my time and money,0.0,complete waste of my time and money
8,the battery life is superb,1.0,the battery life is superb
9,installation was a little bummer,0.0,installation was a little bummer


In [23]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])


In [24]:
!pip install datasets
from datasets import Dataset


In [25]:
train_ds = Dataset.from_pandas(train_df[["clean_text", "label"]])
val_ds   = Dataset.from_pandas(val_df[["clean_text", "label"]])

In [26]:
train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

In [27]:
def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)


Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

In [28]:
train_ds = train_ds.remove_columns(["clean_text"])
val_ds   = val_ds.remove_columns(["clean_text"])


In [29]:
train_ds.set_format("torch")
val_ds.set_format("torch")


In [30]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sentiment_model",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    save_strategy="epoch",
)


In [32]:
import transformers
print(transformers.__version__)


5.8.0


In [33]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }


In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
trainer.train()

In [ ]:
def predict(text):
    clean = preprocess(text)
    inputs = tokenizer(clean, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1).detach().numpy()[0]
    return probs

tests = [
    "bhai kya mast movie hai 🔥",
    "paise waste ho gaye"
]

for t in tests:
    print(t, "→", predict(t))


In [ ]:
train_df["label"].value_counts()


In [ ]:
from collections import Counter

preds = trainer.predict(val_ds)
pred_labels = preds.predictions.argmax(axis=1)

Counter(pred_labels)


In [ ]:
import numpy as np

predictions = trainer.predict(val_ds)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

In [ ]:
val_texts = val_df["text"].values


In [ ]:
errors = []

for text, true, pred in zip(val_texts, y_true, y_pred):
    if true != pred:
        errors.append((text, true, pred))


In [ ]:
false_positives = []
false_negatives = []

for text, true, pred in errors:
    # predicted Positive, actually Negative
    if pred == 1 and true == 0:
        false_positives.append(text)

    # predicted Negative, actually Positive
    if pred == 0 and true == 1:
        false_negatives.append(text)


In [ ]:
print("FALSE POSITIVES (model said Positive, actually Negative):\n")
for t in false_positives:
    print("•", t)

print("\nFALSE NEGATIVES (model said Negative, actually Positive):\n")
for t in false_negatives:
    print("•", t)


In [ ]:
train_df["label"].value_counts()


In [ ]:
neg = train_df[train_df.label == 0]
pos = train_df[train_df.label == 1].sample(len(neg), random_state=42)
train_df = pd.concat([neg, pos]).sample(frac=1, random_state=42)

In [ ]:
model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")